# 1. Análise exploratória dos índices do Pacífico (NOAA) e projeção do ENSO para 2026

## Objetivo
Explorar as séries do NOAA/RONI, identificar anos análogos a 2026 e projetar
os trimestres ASO–NDJ via regressão linear sobre os análogos.

## Fontes de dados
1. **NOAA semanal (2026)** — anomalias TSM nas regiões Niño 1+2, 3, 3.4 e 4.
2. **RONI oficial (1950–2026)** — índice trimestral usado para classificar
   El Niño / La Niña / Neutro e selecionar os análogos históricos.

## Fluxo do notebook
1. Setup e conexões
2. EDA: anomalias por região e mapa das regiões Niño
3. Alvo 2026 (RONI/3.4)
4. Histórico oficial RONI
5. Seleção de anos análogos
6. Modelagem: regressão JAS → ASO/SON/OND/NDJ
7. Validação: LOO, resíduos e previsto vs. observado
8. Resultado final: 2026 observado + projetado

In [8]:
# ============================================================
# 1 Setup: bibliotecas e conexões
# ============================================================
import math
import os
import random
from io import StringIO

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from sqlalchemy import create_engine, inspect, text

# --- Caminho raiz do projeto (usado para salvar CSVs) ---
RAIZ = r"C:\TI\Projetos Univesp\projeto-climaSP"

def carregar_env(caminho):
    if not os.path.exists(caminho):
        return
    with open(caminho, encoding="utf-8") as arquivo:
        for linha in arquivo:
            linha = linha.strip()
            if not linha or linha.startswith("#") or "=" not in linha:
                continue
            chave, valor = linha.split("=", 1)
            os.environ.setdefault(chave.strip(), valor.strip().strip('"').strip("'"))

carregar_env(os.path.join(RAIZ, "Notebookes", ".env"))

<h3>2. Base NOAA semanal (2026) e regiões Niño</h3>

Carregamos o CSV semanal da NOAA com as anomalias de TSM para as quatro
regiões Niño e definimos os metadados geográficos de cada uma.

In [9]:
# --- Conexão com Postgres local (destino dos DataFrames) ---
local_db_url = os.getenv("LOCAL_DB_URL")
if not local_db_url:
    raise RuntimeError(
        "LOCAL_DB_URL não está definida no ambiente. "
        "Confira se Notebookes/.env existe e rode a célula de setup antes desta."
    )
engine_local = create_engine(local_db_url)


In [10]:
# ============================================================
# 2.1 NOAA semanal
# ============================================================
URL_SEMANAL = "https://cpc.ncep.noaa.gov/data/indices/rel_wksst9120.txt"

df_noaa_semanal = pd.read_fwf(
    URL_SEMANAL,
    skiprows=4,
    names=["observation_date", "nino12", "nino3", "nino34", "nino4"],
    colspecs=[(0, 10), (13, 22), (24, 33), (35, 44), (46, 55)],
)
df_noaa_semanal["observation_date"] = pd.to_datetime(
    df_noaa_semanal["observation_date"].str.strip(),
    format="%d%b%Y",
)
df_noaa_semanal = (
    df_noaa_semanal[df_noaa_semanal["observation_date"].dt.year == 2026]
    .reset_index(drop=True)
)
df_noaa_semanal.insert(0, "id", range(1, len(df_noaa_semanal) + 1))
df_noaa_semanal["collected_at"] = pd.Timestamp.now().isoformat(
    sep=" ", timespec="milliseconds"
)
df_noaa_semanal = df_noaa_semanal.set_index("id")

print("Linhas totais:", len(df_noaa_semanal))
print(df_noaa_semanal.dtypes.to_string())
df_noaa_semanal.head()

# ============================================================
# 2.2 Metadados geográficos das regiões Niño
# ============================================================
regioes_descricao = pd.DataFrame({
    "Região":    ["Niño 1+2", "Niño 3", "Niño 3.4", "Niño 4"],
    "Latitude":  ["0°–10° Sul", "5°N–5°S", "5°N–5°S", "5°N–5°S"],
    "Longitude": ["90°W–80°W", "150°W–90°W", "170°W–120°W", "160°E–150°W"],
    "Descrição": [
        "Costa do Peru",
        "Pacífico Leste",
        "Pacífico Centro-Leste (usada para o índice RONI/ONI)",
        "Pacífico Centro-Oeste",
    ],
})
regioes_descricao
df_noaa_semanal

Linhas totais: 41
observation_date    datetime64[us]
nino12                     float64
nino3                      float64
nino34                     float64
nino4                      float64
collected_at                   str


,observation_date,nino12,nino3,nino34,nino4,collected_at
id,,,,,,
1,2026-01-07,-1.0,-1.3,-1.0,-0.4,2026-10-22 16:08:54.980
2,2026-01-14,-0.6,-0.8,-1.0,-0.5,2026-10-22 16:08:54.980
3,2026-01-21,-0.5,-0.4,-0.7,-0.2,2026-10-22 16:08:54.980
4,2026-01-28,-0.2,-0.6,-0.8,-0.3,2026-10-22 16:08:54.980
5,2026-02-04,0.3,-0.5,-0.9,-0.4,2026-10-22 16:08:54.980
6,2026-02-11,0.5,-0.4,-0.6,-0.1,2026-10-22 16:08:54.980
7,2026-02-18,1.0,-0.4,-0.5,0.0,2026-10-22 16:08:54.980
8,2026-02-25,0.7,-0.5,-0.6,-0.1,2026-10-22 16:08:54.980
9,2026-03-04,0.7,-0.3,-0.7,-0.3,2026-10-22 16:08:54.980


<h3>3. Média e percentual das anomalias por região de 2026</h3>

Médias e percentual dos eventos das região Niño (El Niño / La Niña / Neutro),
acima ou abaixo de ±0,5 °C e entre eles.

In [11]:
# ============================================================
# 3. Resumo das anomalias por região (NOAA semanal)
# ============================================================
total = len(df_noaa_semanal)
cols  = ["nino12", "nino3", "nino34", "nino4"]
nomes = {
    "nino12": "Niño 1+2",
    "nino3":  "Niño 3",
    "nino34": "Niño 3.4",
    "nino4":  "Niño 4",
}

linhas = []
for col in cols:
    aquec = (df_noaa_semanal[col] >= 0.5).sum()
    resfr = (df_noaa_semanal[col] <= -0.5).sum()
    linhas.append({
        "Região": nomes[col],
        "Média das anomalias (°C)": df_noaa_semanal[col].mean(),
        "El Niño (>= 0.5°C)":       aquec / total,
        "La Niña (<= -0.5°C)":      resfr / total,
        "Neutro (entre ±0.5°C)":    (total - aquec - resfr) / total,
    })

df_anomalias_anual = (
    pd.DataFrame(linhas)
      .merge(regioes_descricao, on="Região", how="left")
)

percentuais = ["El Niño (>= 0.5°C)", "La Niña (<= -0.5°C)", "Neutro (entre ±0.5°C)"]

df_anomalias_anual.style.format({
    "Média das anomalias (°C)": "{:.2f}",
    **{c: "{:.2%}" for c in percentuais},
})

,Região,Média das anomalias (°C),El Niño (>= 0.5°C),La Niña (<= -0.5°C),Neutro (entre ±0.5°C),Latitude,Longitude,Descrição
0,Niño 1+2,1.94,87.80%,7.32%,4.88%,0°–10° Sul,90°W–80°W,Costa do Peru
1,Niño 3,1.03,58.54%,12.20%,29.27%,5°N–5°S,150°W–90°W,Pacífico Leste
2,Niño 3.4,0.65,53.66%,26.83%,19.51%,5°N–5°S,170°W–120°W,Pacífico Centro-Leste (usada para o índice RONI/ONI)
3,Niño 4,0.18,31.71%,2.44%,65.85%,5°N–5°S,160°E–150°W,Pacífico Centro-Oeste


<h3>3. Série temporal mensal por médias das anomalias por região</h3>

Visualização das quatro regiões Niño, com a anomalia média
do período (cor) e a classificação ONI (El Niño / La Niña / Neutro).

In [12]:
# ============================================================
# 4. Série temporal mensal das anomalias por região
# ============================================================
df_noaa_semanal["observation_date"] = pd.to_datetime(df_noaa_semanal["observation_date"])

media_mensal = (
    df_noaa_semanal
      .groupby(pd.Grouper(key="observation_date", freq="MS"))[
          ["nino12", "nino3", "nino34", "nino4"]
      ]
      .mean()
)

print(media_mensal.head(2))

nomes = {"nino12": "Niño 1+2", "nino3": "Niño 3",
         "nino34": "Niño 3.4", "nino4": "Niño 4"}
cores = {"nino12": "#c0392b", "nino3": "#e67e22",
         "nino34": "#2980b9", "nino4": "#27ae60"}

fig = go.Figure()
for col in ["nino12", "nino3", "nino34", "nino4"]:
    fig.add_trace(go.Scatter(
        x=media_mensal.index,
        y=media_mensal[col],
        name=nomes[col],
        mode="lines+markers",
        line=dict(color=cores[col], width=2.5),
    ))

fig.add_trace(go.Scatter(
    x=[media_mensal.index.min(), media_mensal.index.max()],
    y=[0, 0], mode="lines",
    line=dict(color="red", width=1.5, dash="dash"),
    name=" 0 °C",
))

fig.update_layout(
    title="Evolução mensal das anomalias de TSM — Regiões Niño (2026)",
    xaxis_title="Mês", yaxis_title="Anomalia de TSM (°C)",
    template="plotly_white",
    legend_title="Região",
    hovermode="x unified",
)
fig.show()

                  nino12  nino3  nino34  nino4
observation_date                              
2026-01-01        -0.575 -0.775  -0.875  -0.35
2026-02-01         0.625 -0.450  -0.650  -0.15


<h3>5. Definição e cálculo do RONI/3.4</h3>

<h4> Definição operacional (NOAA)</h4>
- O **ONI** é a média móvel de 3 meses da anomalia de TSM na região **Niño 3.4**
  (5°N–5°S, 120°W–170°W), comparada a uma climatologia de 30 anos.
- Limiares: **≥ +0,5 °C → El Niño**; **≤ −0,5 °C → La Niña**; entre → **Neutro**.
- Um evento oficial exige o limiar mantido por **pelo menos 5 trimestres
  consecutivos**.
- Em 2026 a NOAA passou a usar o **RONI** (Relative Oceanic Niño Index), com os
  mesmos limiares ±0,5 °C, medindo a anomalia do Niño 3.4 em relação à anomalia
  média dos trópicos globais (20°S–20°N).

**Fontes:**
- NOAA Climate.gov — *Climate Variability: Oceanic Niño Index*.
- NOAA CPC — *Oceanic Niño Index (ONI)*.
- Trenberth, K. E. (1997). *The Definition of El Niño*. BAMS 78(12).

In [14]:
# ============================================================
# 6. Cálculo e classificação do RONI/3.4 — 2026
# ============================================================
df_media_mensal = media_mensal.copy()

df_media_mensal["NINO34_MM3"] = (
    df_media_mensal["nino34"].rolling(window=3, min_periods=3).mean()
)

MESES_ABREV = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
temporadas = {
    mes: "".join(MESES_ABREV[(mes - 3 + i) % 12] for i in range(3))
    for mes in range(1, 13)
}
df_media_mensal["Temporada"] = df_media_mensal.index.month.map(temporadas)

def classificar(valor):
    if pd.isna(valor):  return "Sem dados"
    if valor >=  0.5:   return "El Niño"
    if valor <= -0.5:   return "La Niña"
    return "Neutro"

df_media_mensal["Fase"] = df_media_mensal["NINO34_MM3"].apply(classificar)

df_alvo_2026 = (
    df_media_mensal
      .dropna(subset=["NINO34_MM3"])
      [["Temporada", "NINO34_MM3", "Fase"]]
      .copy()
)

print(df_alvo_2026.round({"NINO34_MM3": 2}).to_string(index=False))

ult = df_alvo_2026.iloc[-1]
print(f"> Última temporada: {ult['Temporada']}")
print(f"> Último RONI/3.4: {ult['NINO34_MM3']:+.3f} °C")
print(f"> Fase: {ult['Fase']}")

Temporada  NINO34_MM3    Fase
      JFM       -0.69 La Niña
      FMA       -0.39  Neutro
      MAM       -0.02  Neutro
      AMJ        0.49  Neutro
      MJJ        0.94 El Niño
      JJA        1.38 El Niño
      JAS        1.74 El Niño
      ASO        2.08 El Niño
> Última temporada: ASO
> Último RONI/3.4: +2.085 °C
> Fase: El Niño


In [ ]:
# ============================================================
# 7 — Gráfico do RONI/3.4 por temporada
# ============================================================

fig = go.Figure()

# Série do RONI/3.4
fig.add_trace(go.Scatter(
    x=df_alvo_2026["Temporada"],
    y=df_alvo_2026["NINO34_MM3"],
    mode="lines+markers",
    name="RONI/3.4 — Niño 3.4",
    line=dict(width=2.5),
    marker=dict(size=8)
))

# Limite de El Niño
fig.add_hline(
    y=0.5,
    line_dash="dash",
    annotation_text="+0,5 °C"
)

# Limite de La Niña
fig.add_hline(
    y=-0.5,
    line_dash="dash",
    annotation_text="-0,5 °C"
)

# Linha de referência 0 °C
fig.add_hline(
    y=0,
    line_dash="dot"
)

fig.update_layout(
    title="RONI/3.4 — Niño 3.4 por temporadas de 3 meses - médias móveis sobrepostas",
    legend_title="Fases do ENSO (El Niño, La Niña, Neutro)",
    xaxis_title="Temporada",
    yaxis_title="RONI/3.4 (°C)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

<h3> 7. Histórico oficial do RONI (1950–2026)</h3>

Baixando a série histórica trimestral do RONI direto do CPC/NOAA e
adicionamos uma coluna `data` com o **mês central** de cada trimestre.
Isso permite ordenar cronologicamente e cruzar com a série mensal
quando necessário.

Mapeamento do mês central:
- **DJF** → jan  ·  **JFM** → fev  ·  **FMA** → mar  ·  **MAM** → abr
- **AMJ** → mai  ·  **MJJ** → jun  ·  **JJA** → jul  ·  **JAS** → ago
- **ASO** → set  ·  **SON** → out  ·  **OND** → nov  ·  **NDJ** → dez

In [21]:
# ============================================================
# 7. Download do RONI oficial (1950–JAS/2026) + ASO/2026 calculado
# ============================================================
url_roni = "https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt"
resp = requests.get(url_roni)
resp.raise_for_status()

df_roni_raw = pd.read_csv(StringIO(resp.text), sep=r"\s+")

seas_2026_oficial = ["DJF", "JFM", "FMA", "MAM", "AMJ", "MJJ", "JJA", "JAS"]

df_roni_hist = df_roni_raw[
    (df_roni_raw["YR"] <= 2025)
    | ((df_roni_raw["YR"] == 2026) & (df_roni_raw["SEAS"].isin(seas_2026_oficial)))
].copy()


# ============================================================
# 7.1 Converte SEAS em data (mês central do trimestre)
# ============================================================
meses_centrais = {
    "DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
    "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12,
}

df_roni_hist["mes_central"] = df_roni_hist["SEAS"].map(meses_centrais)
df_roni_hist["data"] = pd.to_datetime(
    df_roni_hist["YR"].astype(str) + "-"
    + df_roni_hist["mes_central"].astype(str) + "-01"
)

# ============================================================
# 7.2 Concatena o ASO/2026 calculado da série semanalNa 
# ============================================================

aso_2026 = df_alvo_2026.loc[
    df_alvo_2026["Temporada"] == "ASO", "NINO34_MM3"
]
if aso_2026.empty:
    raise ValueError("ASO/2026 não encontrado em df_alvo_2026")

df_roni_hist = pd.concat(
    [
        df_roni_hist,
        pd.DataFrame([{
            "SEAS": "ASO",
            "YR": 2026,
            "ANOM": float(aso_2026.iloc[-1]),
            "mes_central": meses_centrais["ASO"],
            "data": pd.Timestamp("2026-09-01"),
        }]),
    ],
    ignore_index=True,
)

print(f"Total de registros: {len(df_roni_hist)}")
print(f"Colunas: {df_roni_hist.columns.tolist()}")
print(f"Ano mínimo: {df_roni_hist['YR'].min()}")
print(f"Ano máximo: {df_roni_hist['YR'].max()}")
print(df_roni_hist.tail(3).to_string(index=False))


Total de registros: 921
Colunas: ['SEAS', 'YR', 'ANOM', 'mes_central', 'data']
Ano mínimo: 1950
Ano máximo: 2026
SEAS   YR  ANOM  mes_central       data
 JJA 2026 1.360            7 2026-07-01
 JAS 2026 1.690            8 2026-08-01
 ASO 2026 2.085            9 2026-09-01


<h3> 8. Fases ENSO no histórico (1950–2026)</h3>

Aplicamos o limiar ±0,5 °C a cada trimestre do histórico para contar
quantos trimestres caíram em El Niño, La Niña ou Neutro, e plotamos
a série colorida por fase para inspeção visual.

In [15]:
# ============================================================
# 8. Classificação das fases no histórico
# ============================================================
def classificar(valor):
    if valor >=  0.5: return "El Niño"
    if valor <= -0.5: return "La Niña"
    return "Neutro"

df_roni_hist["fase"] = df_roni_hist["ANOM"].apply(classificar)
print(df_roni_hist["fase"].value_counts())

# ============================================================
# 8.1 Série histórica colorida por fase
# ============================================================
fig = px.line(
    df_roni_hist,
    x="data", y="ANOM", color="fase",
    title="RONI — Série temporal (1950–2026)",
    labels={"data": "Ano", "ANOM": "RONI (°C)", "fase": "Fase"},
    color_discrete_map={
        "El Niño": "#d62728",
        "La Niña": "#1f77b4",
        "Neutro":  "gray",
    },
)

fig.add_hline(y= 0.5, line_dash="dash", line_color="red",  opacity=0.5)
fig.add_hline(y=-0.5, line_dash="dash", line_color="blue", opacity=0.5)
fig.add_hline(y= 0,   line_dash="dot",  line_color="black", opacity=0.3)

fig.update_layout(
    height=500,
    hovermode="x unified",
    xaxis_title="Ano",
    yaxis_title="RONI (°C)",
)
fig.show()

fase
Neutro     444
El Niño    241
La Niña    236
Name: count, dtype: int64


In [ ]:
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

ordem_seas = df_alvo_2026["Temporada"].astype(str).tolist()

# 1)
y_2026 = (
    df_alvo_2026
      .set_index("Temporada")
      .reindex(ordem_seas)["NINO34_MM3"]
      .values
)

# 2) Referência para padronização (climatologia externa, não a janela)
#    Ex.: usar a média/desvio de todos os anos 1991–2020 nas mesmas temporadas
clim = df_roni_hist[df_roni_hist["YR"].between(1991, 2020)] \
         .groupby("SEAS")["ANOM"].agg(["mean", "std"])
mu  = clim.loc[ordem_seas, "mean"].values
sig = clim.loc[ordem_seas, "std"].values

y_2026_z = (y_2026 - mu) / sig

# 3) Histórico
historicos = {}
for ano, grupo in df_roni_hist.groupby("YR"):
    if ano == 2026:
        continue
    if 2023 <= ano <= 2025:      # evita autocorrelação trivial
        continue
    g = grupo.set_index("SEAS").reindex(ordem_seas)
    if g["ANOM"].notna().all():
        historicos[ano] = g["ANOM"].values

# 4) Métricas no espaço z
linhas = []
for ano, seq in historicos.items():
    seq_z = (seq - mu) / sig
    r, p  = pearsonr(seq_z, y_2026_z)
    rmse  = float(np.sqrt(np.mean((seq_z - y_2026_z) ** 2)))
    linhas.append({"Ano": ano, "r": r, "p": p, "RMSE_z": rmse})

df = pd.DataFrame(linhas)

# 5) Correção de múltiplas comparações (FDR)
rej, p_adj, _, _ = multipletests(df["p"], alpha=0.05, method="fdr_bh")
df["p_adj"]      = p_adj
df["significante"] = rej

# 6) Rank por correlação (forma), desempate por RMSE_z
df = df.sort_values(["r", "RMSE_z"], ascending=[False, True]).reset_index(drop=True)

print(df.head(10).to_string(index=False))


<h3>9. Comparação de 2026 com cada ano histórico</h3>

Para cada ano de 1950 a 2025, extraímos a janela **JFM–ASO** (8 trimestres)
e comparamos com o mesmo intervalo em 2026 usando quatro métricas:

- **RMSE** — erro médio absoluto em °C (menor é melhor).
- **Correlação** — similaridade de forma da curva (maior é melhor).
- **Amplitude** — range total do ano (evita análogos "achatados").
- **Trim. divergentes** — quantos trimestres diferem mais de 0,5 °C.

Ordenamos por RMSE para ranquear os candidatos.


In [22]:
# ============================================================
# 9. Comparação de cada ano histórico com 2026
# ============================================================
ordem_seas = df_alvo_2026["Temporada"].astype(str).tolist()

# Alvo: sequência de 2026 na ordem correta
y_2026 = (
    df_alvo_2026
      .set_index("Temporada")
      .reindex(ordem_seas)["NINO34_MM3"]
      .values
)

print("Alvo (2026):")
for t, v in zip(ordem_seas, y_2026):
    print(f"  {t}: {v:+.3f}")

# Histórico — janelas JFM..ASO por ano (exceto 2026)
historicos = {}
for ano, grupo in df_roni_hist.groupby("YR"):
    if ano == 2026:
        continue
    g = grupo.set_index("SEAS").reindex(ordem_seas)
    if g["ANOM"].notna().all():
        historicos[ano] = g["ANOM"].values

print(f"\nAnos com janela completa JFM–ASO: {len(historicos)}")

# Métricas
resultados = []
for ano, seq in historicos.items():
    rmse      = float(np.sqrt(np.mean((seq - y_2026) ** 2)))
    corr      = float(np.corrcoef(seq, y_2026)[0, 1])
    std       = float(np.std(seq, ddof=1))
    amplitude = float(seq.max() - seq.min())
    trim_div  = int(sum(abs(s - a) > 0.5 for s, a in zip(seq, y_2026)))
    aso       = float(seq[-1])

    resultados.append({
        "Ano": ano,
        **{t: round(v, 2) for t, v in zip(ordem_seas, seq)},
        "RMSE": round(rmse, 3),
        "Correlacao": round(corr, 3),
        "Std": round(std, 3),
        "Amplitude": round(amplitude, 3),
        "trim_divergente": trim_div,
        "ASO": round(aso, 3),
    })

df_rank = (
    pd.DataFrame(resultados)
      .sort_values("Ano")
      .reset_index(drop=True)
)

print("\n===== TOP 10 ANOS MAIS PARECIDOS COM 2026 (JFM–ASO) =====")
print(df_rank.sort_values("RMSE").head(10).to_string(index=False))


Alvo (2026):
  JFM: -0.692
  FMA: -0.387
  MAM: -0.020
  AMJ: +0.488
  MJJ: +0.935
  JJA: +1.377
  JAS: +1.745
  ASO: +2.085

Anos com janela completa JFM–ASO: 76

===== TOP 10 ANOS MAIS PARECIDOS COM 2026 (JFM–ASO) =====
 Ano   JFM   FMA   MAM   AMJ  MJJ  JJA  JAS  ASO  RMSE  Correlacao   Std  Amplitude  trim_divergente
1997 -0.03  0.17  0.47  0.83 1.20 1.50 1.84 2.07 0.387       1.000 0.775       2.10                2
1965  0.10  0.22  0.36  0.64 1.01 1.33 1.49 1.76 0.409       0.995 0.625       1.66                2
1972  0.03  0.32  0.51  0.66 0.87 1.14 1.40 1.64 0.462       0.993 0.549       1.61                3
1982  0.22  0.23  0.46  0.74 0.84 0.89 1.14 1.64 0.538       0.962 0.479       1.42                3
1963 -0.12  0.04  0.06  0.01 0.37 0.73 1.08 1.07 0.608       0.945 0.491       1.20                5
2015  0.47  0.59  0.72  0.89 1.02 1.27 1.54 1.80 0.628       0.984 0.468       1.33                3
2023 -0.68 -0.55 -0.31 -0.04 0.28 0.54 0.79 1.04 0.664       0.999 0.63

<h3>10. Seleção final dos análogos</h3>

Aplicamos filtros para ficar apenas com anos que:
- têm alta correlação com 2026 (**r ≥ 0.90**),
- têm RMSE baixo (**≤ 0.75**),
- têm pouca divergência trimestral (**≤ 5 trimestres**),
- têm amplitude mínima (**≥ 0.60**) — evita anos muito "achatados",
- têm ASO observado suficientemente alto (**≥ 0.79 °C**).

Esses são os análogos que alimentarão a regressão ASO → SON/OND/NDJ.


In [23]:
# ============================================================
# 10. Filtro dos análogos
# ============================================================
LIMIAR_CORR  = 0.900
LIMIAR_RMSE  = 0.750
LIMIAR_TRIM  = 5
LIMIAR_AMPL  = 0.60
LIMIAR_ASO   = 0.79
ultima_seas = ordem_seas[-1]

df_analogos = (
    df_rank[
        (df_rank["Correlacao"]       >= LIMIAR_CORR) &
        (df_rank["RMSE"]             <= LIMIAR_RMSE) &
        (df_rank["trim_divergente"]  <= LIMIAR_TRIM) &
        (df_rank["Amplitude"]        >= LIMIAR_AMPL) &
        (df_rank[ultima_seas]        >= LIMIAR_ASO)
    ]
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print(f"Análogos selecionados: {len(df_analogos)}")
print(df_analogos[["Ano", "RMSE", "Correlacao", "Amplitude", "trim_divergente", ultima_seas]].to_string(index=False))

# ============================================================
# 10.1 Plot: análogos + média + 2026
# ============================================================
mat_med = df_analogos[ordem_seas].values.astype(float)
media_an = mat_med.mean(axis=0)

fig = go.Figure()

for _, row in df_analogos.sort_values("Ano").iterrows():
    seq = [row[t] for t in ordem_seas]
    fig.add_trace(go.Scatter(
        x=ordem_seas, y=seq,
        mode="lines+markers",
        name=f'{int(row["Ano"])}',
        line=dict(width=1.8),
        opacity=0.85,
        hovertemplate=(
            f'<b>{int(row["Ano"])}</b><br>'
            '%{x} = %{y:.3f}<br>'
            f'Correlação: {row["Correlacao"]:.3f}<br>'
            f'RMSE: {row["RMSE"]:.3f}'
            '<extra></extra>'
        ),
    ))

fig.add_trace(go.Scatter(
    x=ordem_seas, y=media_an,
    mode="lines+markers",
    name="Média dos análogos",
    line=dict(width=2.5, color="#1f77b4", dash="dash"),
    marker=dict(size=8, color="#1f77b4"),
))

fig.add_trace(go.Scatter(
    x=ordem_seas, y=y_2026,
    mode="lines+markers",
    name="2026 (alvo)",
    line=dict(width=3.5, color="#000000"),
    marker=dict(size=9, color="#000000"),
))

fig.add_hline(y= 0.5, line_dash="dot", line_color="#d62728", opacity=0.5)
fig.add_hline(y=-0.5, line_dash="dot", line_color="#1f77b4", opacity=0.5)
fig.add_hline(y= 0,   line_dash="dot", line_color="#7f7f7f", opacity=0.3)

fig.update_layout(
    title=f"Análogos históricos de 2026 (r ≥ {LIMIAR_CORR}) — JFM a ASO",
    xaxis_title="Temporada",
    yaxis_title="NINO34_MM3 (°C)",
    template="plotly_white",
    hovermode="closest",
    legend_title="Ano",
)
fig.show()


Análogos selecionados: 10
 Ano  RMSE  Correlacao  Amplitude  trim_divergente  ASO
1997 0.387       1.000       2.10                2 2.07
1965 0.409       0.995       1.66                2 1.76
1972 0.462       0.993       1.61                3 1.64
1982 0.538       0.962       1.42                3 1.64
1963 0.608       0.945       1.20                5 1.07
2015 0.628       0.984       1.33                3 1.80
2023 0.664       0.999       1.72                5 1.04
2002 0.677       0.980       0.96                4 0.97
1976 0.700       0.991       1.54                4 0.89
1957 0.711       0.910       0.92                5 1.14


<h3>11. Modelagem: regressão→ trimestres futuros</h3>

A ideia é simples: cada trimestre futuro (SON, OND e NDJ) é estimado
como função linear do valor de **ASO** do mesmo ano:

> `Y_futuro = a · x + b`

Ajustamos **uma regressão por trimestre** (três retas independentes),
usando os anos análogos como conjunto de treino e o ASO de 2026
(**+2.085 °C**, último observado) como entrada para gerar as previsões.


In [24]:
# ============================================================
# 11. Modelagem — regressão por trimestre futuro
# ============================================================
anos_analogos = df_analogos["Ano"].tolist()
print(f"Análogos usados: {anos_analogos}")

seas_futuro = ["SON", "OND", "NDJ"]

# --- Matrizes de treino ---
X_train = []
Y_train = []
for ano in anos_analogos:
    hist = df_roni_hist[df_roni_hist["YR"] == ano].set_index("SEAS")
    X_train.append(hist.loc["ASO", "ANOM"])
    Y_train.append([hist.loc[s, "ANOM"] for s in seas_futuro])

X_train = np.array(X_train)          # shape (n_análogos,)
Y_train = np.array(Y_train)          # shape (n_análogos, 3)

# --- Valor-alvo ---
x_alvo = y_2026[-1]                   # ASO de 2026
print(f"\nASO de 2026 = {x_alvo:+.3f}")

# --- Ajuste por trimestre ---
previsoes = []
coeficientes = []
for i, seas in enumerate(seas_futuro):
    a, b = np.polyfit(X_train, Y_train[:, i], deg=1)
    y_prev = a * x_alvo + b
    previsoes.append(y_prev)
    coeficientes.append((seas, a, b, y_prev))
    print(f"{seas}: y = {a:.3f} * x + {b:.3f}  →  2026: {y_prev:+.3f} °C")

# --- DataFrame de saída ---
df_previsao = pd.DataFrame({
    "Temporada": seas_futuro,
    "Previsão":  np.round(previsoes, 3),
})
df_previsao["Fase"] = np.where(
    df_previsao["Previsão"] >=  0.5, "El Niño",
    np.where(df_previsao["Previsão"] <= -0.5, "La Niña", "Neutro")
)

print("\n===== PROJEÇÃO DE 2026 (SON–NDJ) =====")
print(df_previsao.to_string(index=False))


Análogos usados: [1997, 1965, 1972, 1982, 1963, 2015, 2023, 2002, 1976, 1957]

ASO de 2026 = +2.085
SON: y = 1.062 * x + 0.081  →  2026: +2.295 °C
OND: y = 1.066 * x + 0.205  →  2026: +2.427 °C
NDJ: y = 1.068 * x + 0.204  →  2026: +2.430 °C

===== PROJEÇÃO DE 2026 (SON–NDJ) =====
Temporada  Previsão    Fase
      SON     2.295 El Niño
      OND     2.427 El Niño
      NDJ     2.430 El Niño


<h3>12. Validação numérica da regressão</h3>

Com apenas 10 análogos, o R² do ajuste é quase sempre alto — por isso
o teste mais honesto é o **leave-one-out (LOO)**: treinamos com 9
análogos, prevemos o 10º, repetimos 10 vezes e medimos o erro real.

Comparamos também **RMSE do ajuste** vs **RMSE do LOO** para detectar
overfitting. Se o gap for grande, a regressão está "decorando" os
análogos.


In [25]:
# ============================================================
# 12. Validação numérica (ajuste + leave-one-out)
# ============================================================
resultados_fit = []

for i, seas in enumerate(seas_futuro):
    y = Y_train[:, i]

    # --- ajuste normal ---
    a, b  = np.polyfit(X_train, y, deg=1)
    y_hat = a * X_train + b
    resid = y - y_hat

    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2       = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse_fit = np.sqrt(np.mean(resid ** 2))

    # --- leave-one-out ---
    erros_loo = []
    for j in range(len(X_train)):
        mask = np.arange(len(X_train)) != j
        a_j, b_j = np.polyfit(X_train[mask], y[mask], deg=1)
        y_pred_j = a_j * X_train[j] + b_j
        erros_loo.append(y[j] - y_pred_j)
    rmse_loo = float(np.sqrt(np.mean(np.array(erros_loo) ** 2)))

    resultados_fit.append({
        "Temporada":   seas,
        "a":           a,
        "b":           b,
        "R2":          r2,
        "RMSE_ajuste": rmse_fit,
        "RMSE_LOO":    rmse_loo,
        "gap_LOO":     rmse_loo - rmse_fit,
    })

df_fit = pd.DataFrame(resultados_fit)
print(df_fit.round(3).to_string(index=False))


Temporada     a     b    R2  RMSE_ajuste  RMSE_LOO  gap_LOO
      SON 1.062 0.081 0.940        0.108     0.130    0.022
      OND 1.066 0.205 0.898        0.144     0.181    0.037
      NDJ 1.068 0.204 0.790        0.221     0.266    0.045


<h3>13. Validação visual da regressão</h3>

Três gráficos diagnósticos, um para cada tipo de verificação:

1. **Scatter + reta** — mostra o ajuste por trimestre (ASO → SON/OND/NDJ).
   Pontos colados na reta indicam relação linear forte.
2. **Resíduos** — diferença entre observado e previsto em cada ano. O
   ideal é que fiquem pequenos e **sem padrão**. Estrutura em "U"
   denuncia curvatura não capturada.
3. **Previsto vs. Observado** — cada ponto é `(previsto, observado)`. Se
   o modelo fosse perfeito, tudo cairia na diagonal `y = x`. Afastamentos
   sistemáticos revelam viés.


In [27]:
# ============================================================
# 13.1 Scatter + reta por trimestre
# ============================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=seas_futuro)

for i, seas in enumerate(seas_futuro):
    row, col = divmod(i, 2)
    y = Y_train[:, i]
    a, b = np.polyfit(X_train, y, deg=1)
    x_reta = np.linspace(X_train.min() - 0.1, X_train.max() + 0.1, 50)

    fig.add_trace(go.Scatter(
        x=X_train, y=y,
        mode="markers+text",
        text=[str(an) for an in anos_analogos],
        textposition="top center",
        marker=dict(size=10, color="#1f77b4"),
        showlegend=False,
        hovertemplate="Ano %{text}<br>ASO=%{x:.3f}<br>" + seas + "=%{y:.3f}<extra></extra>",
    ), row=row+1, col=col+1)

    fig.add_trace(go.Scatter(
        x=x_reta, y=a * x_reta + b,
        mode="lines",
        line=dict(color="#d62728", width=2),
        showlegend=False,
    ), row=row+1, col=col+1)

    fig.add_trace(go.Scatter(
        x=[x_alvo], y=[a * x_alvo + b],
        mode="markers",
        marker=dict(size=14, color="#d62728", symbol="star"),
        showlegend=False,
        hovertemplate=f"2026 previsto {seas}=%{{y:.3f}}<extra></extra>",
    ), row=row+1, col=col+1)

fig.update_layout(
    height=700,
    title="Validação — ASO → SON/OND/NDJ",
    template="plotly_white",
)
fig.show()


# ============================================================
# 13.2 Resíduos por trimestre
# ============================================================
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f"{s} — resíduos" for s in seas_futuro])

for i, seas in enumerate(seas_futuro):
    row, col = divmod(i, 2)
    y = Y_train[:, i]
    a, b = np.polyfit(X_train, y, deg=1)
    resid = y - (a * X_train + b)

    fig.add_trace(go.Scatter(
        x=X_train, y=resid,
        mode="markers+text",
        text=[str(an) for an in anos_analogos],
        textposition="top center",
        marker=dict(size=10, color="#ff7f0e"),
        showlegend=False,
        hovertemplate="Ano %{text}<br>resíduo=%{y:.3f}<extra></extra>",
    ), row=row+1, col=col+1)

    fig.add_hline(y=0, line_dash="dash", row=row+1, col=col+1)

fig.update_layout(
    height=700,
    title="Resíduos da regressão (ideal: pequenos e aleatórios)",
    template="plotly_white",
)
fig.show()


# ============================================================
# 13.3 Previsto vs. Observado
# ============================================================
fig = go.Figure()

for i, seas in enumerate(seas_futuro):
    y = Y_train[:, i]
    a, b = np.polyfit(X_train, y, deg=1)
    y_hat = a * X_train + b

    fig.add_trace(go.Scatter(
        x=y_hat, y=y,
        mode="markers+text",
        name=seas,
        marker=dict(size=10),
        text=[str(an) for an in anos_analogos],
        textposition="top center",
        hovertemplate="Ano %{text}<br>previsto=%{x:.3f}<br>observado=%{y:.3f}<extra></extra>",
    ))

pred_all = []
obs_all  = []
for i in range(len(seas_futuro)):
    y = Y_train[:, i]
    a, b = np.polyfit(X_train, y, deg=1)
    pred_all.extend(a * X_train + b)
    obs_all.extend(y)

lo = min(min(pred_all), min(obs_all)) - 0.2
hi = max(max(pred_all), max(obs_all)) + 0.2

fig.add_trace(go.Scatter(
    x=[lo, hi], y=[lo, hi],
    mode="lines",
    line=dict(color="black", dash="dash"),
    name="y = x (ideal)",
))

fig.update_layout(
    title="Previsto vs. Observado (quanto mais perto da diagonal, melhor)",
    xaxis_title="Previsto",
    yaxis_title="Observado",
    template="plotly_white",
)
fig.show()


<h3> 14. Resultado final — 2026 observado + projetado</h3>

Consolidamos num único DataFrame os **8 trimestres observados** de 2026
(JFM a ASO, vindos do RONI) com os **3 trimestres previstos** pela
regressão (SON a NDJ), e plotamos a série completa com a distinção
visual entre observado e projetado.


In [28]:
# ============================================================
# 14.1 Gráfico: 2026 observado + projeção SON–NDJ
# ============================================================
fig = go.Figure()

# --- série observada ---
fig.add_trace(go.Scatter(
    x=df_alvo_2026["Temporada"],
    y=df_alvo_2026["NINO34_MM3"],
    mode="lines+markers",
    name="2026 — observado",
    line=dict(width=2.5, color="#000000"),
    marker=dict(size=8, color="#000000"),
))

# --- projeção ---
x_proj = [df_alvo_2026["Temporada"].iloc[-1]] + seas_futuro
y_proj = [df_alvo_2026["NINO34_MM3"].iloc[-1]] + previsoes

fig.add_trace(go.Scatter(
    x=x_proj, y=y_proj,
    mode="lines+markers",
    name="2026 — projetado (regressão)",
    line=dict(width=2.5, color="#d62728", dash="dash"),
    marker=dict(size=9, color="#d62728", symbol="diamond"),
    hovertemplate="Temporada: %{x}<br>RONI/3.4: %{y:.2f} °C<extra></extra>",
))

fig.add_hline(y= 0.5, line_dash="dash", annotation_text="+0,5 °C")
fig.add_hline(y=-0.5, line_dash="dash", annotation_text="-0,5 °C")
fig.add_hline(y= 0,   line_dash="dot")

fig.update_layout(
    title="RONI/3.4 — 2026 observado + projeção SON–NDJ",
    legend_title="Série",
    xaxis_title="Temporada",
    yaxis_title="RONI/3.4 (°C)",
    template="plotly_white",
    hovermode="x unified",
)
fig.show()


# ============================================================
# 14.2 DataFrame consolidado 2026 (observado + previsto)
# ============================================================
temporadas_2026 = ["JFM","FMA","MAM","AMJ","MJJ","JJA","JAS","ASO",
                   "SON","OND","NDJ"]

# --- observado (JFM..ASO) ---
df_obs = df_alvo_2026[["Temporada", "NINO34_MM3", "Fase"]].copy()
df_obs["Ano"]    = 2026
df_obs["Origem"] = "Observado"

# --- previsto (SON..NDJ) ---
df_prev = df_previsao.rename(columns={"Previsão": "NINO34_MM3"}).copy()
df_prev["Ano"]    = 2026
df_prev["Origem"] = "Previsto"

# --- concatenação ---
df_2026 = pd.concat([df_obs, df_prev], ignore_index=True)
df_2026["NINO34_MM3"] = df_2026["NINO34_MM3"].round(3)
df_2026["Temporada"]  = pd.Categorical(
    df_2026["Temporada"],
    categories=temporadas_2026,
    ordered=True,
)
df_2026 = df_2026.sort_values("Temporada").reset_index(drop=True)

df_2026


,Temporada,NINO34_MM3,Fase,Ano,Origem
0,JFM,-0.692,La Niña,2026,Observado
1,FMA,-0.387,Neutro,2026,Observado
2,MAM,-0.020,Neutro,2026,Observado
3,AMJ,0.488,Neutro,2026,Observado
4,MJJ,0.935,El Niño,2026,Observado
5,JJA,1.377,El Niño,2026,Observado
6,JAS,1.745,El Niño,2026,Observado
7,ASO,2.085,El Niño,2026,Observado
8,SON,2.295,El Niño,2026,Previsto
9,OND,2.427,El Niño,2026,Previsto


<h3> 15. Persistência no Postgres local </h3>

Todos os DataFrames produzidos no notebook são persistidos no banco
local. Usamos `if_exists="replace"` para **reconstruir** cada tabela
a cada execução — assim evitamos duplicatas quando o notebook é
reprocessado célula por célula.

A função `salvar()` padroniza:
- reset do índice (começa em 1);
- nome do index label;
- modo `multi` (insere em lotes).

In [29]:
# ============================================================
# 15. Persistência no Postgres local
# ============================================================
def salvar(df, nome, index_label="index"):
    df = df.copy()
    df.index = range(1, len(df) + 1)
    df.to_sql(
        name=nome,
        con=engine_local,
        if_exists="replace",
        index=True,
        index_label=index_label,
        method="multi",
    )
    print(f"✔ {nome:<20} {len(df):>4} linhas")


In [30]:
# ============================================================
# 15.1 Tabelas do notebook
# ============================================================
salvar(df_noaa_semanal,   "df_noaa_weekly")
salvar(df_anomalias_anual, "df_anomalia_anual")
salvar(media_mensal.reset_index(), "df_media_mensal")
salvar(df_alvo_2026,      "df_roni")
salvar(df_roni_hist,      "df_hist_roni")
salvar(df_rank,           "df_rank")
salvar(df_analogos,       "df_analogos")
salvar(df_previsao,       "df_proj")
salvar(df_2026,           "df_2026")

✔ df_noaa_weekly         41 linhas
✔ df_anomalia_anual       4 linhas
✔ df_media_mensal        10 linhas
✔ df_roni                 8 linhas
✔ df_hist_roni          921 linhas
✔ df_rank                76 linhas
✔ df_analogos            10 linhas
✔ df_proj                 3 linhas
✔ df_2026                11 linhas


In [31]:
TABELAS = [
    "df_noaa_weekly",
    "df_anomalia_anual",
    "df_media_mensal",
    "df_hist_roni",
    "df_rank",
    "df_analogos",
    "df_2026",
]


In [32]:
# --- Conexão com Supabase (publicação) ---
supabase_db_url = os.getenv("SUPABASE_DB_URL")
if not supabase_db_url:
    raise RuntimeError(
        "SUPABASE_DB_URL não está definida no ambiente. "
        "Confira se Notebookes/.env existe e rode a célula de setup antes desta."
    )
engine_claud = create_engine(supabase_db_url)


In [33]:
def bd_cloud_spb(nome, origem=engine_local, destino=engine_claud):
    df = pd.read_sql(text(f'SELECT * FROM "{nome}"'), origem)
    df.to_sql(
        name=nome,
        con=destino,
        if_exists="replace",
        index=False,
        method="multi",
    )
    n = pd.read_sql(text(f'SELECT count(*) AS n FROM "{nome}"'), destino)["n"][0]
    print(f"{nome:<22} local={len(df):>6}  supabase={n:>6}")

for nome in TABELAS:
    bd_cloud_spb(nome)

df_noaa_weekly         local=    41  supabase=    41
df_anomalia_anual      local=     4  supabase=     4
df_media_mensal        local=    10  supabase=    10
df_hist_roni           local=   921  supabase=   921
df_rank                local=    76  supabase=    76
df_analogos            local=    10  supabase=    10
df_2026                local=    11  supabase=    11


In [34]:
# ============================================================
# 5b. Mapa das regiões Niño — versão Folium
# ============================================================
import folium
from branca.colormap import LinearColormap

# --- médias do período (mesmo cálculo da versão Plotly) ---
medias = df_noaa_semanal[["nino12", "nino3", "nino34", "nino4"]].mean()

def class_oni(v):
    if v >=  0.5: return "El Niño"
    if v <= -0.5: return "La Niña"
    return "Neutro"

# --- mesma tabela de regiões ---
regioes = [
    ("Niño 1+2", "nino12", (-10, 0), (-90, -80)),
    ("Niño 3",   "nino3",  (-5, 5),  (-150, -90)),
    ("Niño 3.4", "nino34", (-5, 5),  (-170, -120)),
    ("Niño 4",   "nino4",  (-5, 5),  (160, -150)),
]

# --- colormap divergente (azul ↔ vermelho) ---
vmax = max(1.0, max(abs(medias[c]) for _, c, _, _ in regioes))
colormap = LinearColormap(
    colors=["#1565c0", "#ececec", "#cb182b"],
    vmin=-vmax, vmax=vmax,
    caption="Anomalia média SST (°C)",
)

def cor(v):
    return colormap(v)

# --- mapa base ---
m = folium.Map(
    location=[-2, -140],
    zoom_start=3,
    tiles="CartoDB positron",   # mais limpo que OSM para dados
)

# --- desenha cada região ---
for nome, col, lat, lon in regioes:
    v = medias[col]
    cls = class_oni(v)

    lat0, lat1 = lat
    lon0, lon1 = lon
    if lon0 > lon1:
        lon1 += 360

    # retângulo geográfico (borda + preenchimento)
    folium.Rectangle(
        bounds=[[lat0, lon0], [lat1, lon1]],
        color=cor(v),
        weight=2.5,
        fill=True,
        fillColor=cor(v),
        fillOpacity=0.25,
        tooltip=f"{nome}: {v:+.2f} °C | {cls}",
        popup=folium.Popup(
            f"""<b>{nome}</b><br>
                Anomalia média SST: <b>{v:+.2f} °C</b><br>
                Classificação ONI: <b>{cls}</b>""",
            max_width=250,
        ),
    ).add_to(m)

    # rótulo central
    latc, lonc = (lat0 + lat1) / 2, (lon0 + lon1) / 2
    folium.Marker(
        location=[latc, lonc],
        icon=folium.DivIcon(
            html=f"""<div style="font-size:13px; font-weight:bold;
                     color:#0d1b2a; text-align:center; white-space:nowrap;
                     text-shadow: 1px 1px 2px white;">
                     {nome}<br>{v:+.2f}°C ({cls})</div>""",
            icon_size=(160, 40),
            icon_anchor=(80, 20),
        ),
    ).add_to(m)

# --- legenda (colormap) ---
colormap.add_to(m)

m

c:\TI\Projetos Univesp\projeto-climaSP\.venv\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
